# 06 Export Artifacts for Streamlit


In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path('..')/'src'))


## Objectives

- Smoke-test the end-to-end pipeline used by the Streamlit app.
- Ensure artifacts exist and can be loaded.
- Validate that recommendations work with filters.


In [2]:
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
from scipy import sparse

models_dir = Path('..')/'models'

vectorizer = joblib.load(models_dir/'tfidf_vectorizer.joblib')
tfidf_matrix = sparse.load_npz(models_dir/'tfidf_matrix.npz')
index_df = pd.read_parquet(models_dir/'course_index.parquet')

bundle = joblib.load(models_dir/'profit_model.joblib')
pipe = bundle['pipeline']
num_cols = bundle['numeric_cols']
cat_cols = bundle['categorical_cols']

(index_df.shape, tfidf_matrix.shape)


((3677, 15), (3677, 3291))

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend(query: str, top_k: int = 10, subject=None, max_price=None, is_paid=None, alpha: float = 0.65):
    qv = vectorizer.transform([query])
    sims = cosine_similarity(qv, tfidf_matrix).ravel()

    cand = index_df.copy()
    cand['similarity'] = sims

    if subject is not None:
        cand = cand[cand['subject'] == subject]
    if max_price is not None:
        cand = cand[cand['price'] <= max_price]
    if is_paid is not None:
        cand = cand[cand['is_paid'] == is_paid]

    # Predict profit
    # Ensure derived features expected by the model exist (e.g., 'title_len')
    if 'title_len' not in cand.columns and 'course_title' in cand.columns:
        cand['title_len'] = cand['course_title'].astype(str).str.len()

    X = cand[num_cols + cat_cols]
    pred_log = pipe.predict(X)
    cand['predicted_profit'] = np.expm1(pred_log)

    # Hybrid rerank
    def _minmax(s):
        lo, hi = float(s.min()), float(s.max())
        if hi - lo < 1e-12:
            return np.zeros(len(s))
        return (s - lo) / (hi - lo)

    cand['_sim'] = _minmax(cand['similarity'].astype(float))
    cand['_pp']  = _minmax(cand['predicted_profit'].astype(float))
    cand['score'] = alpha * cand['_sim'] + (1 - alpha) * cand['_pp']

    cols = ['course_title','subject','level','price','similarity','predicted_profit','score','url']
    return cand.sort_values('score', ascending=False).head(top_k)[cols]

recommend('python data science', subject='Web Development', max_price=100, top_k=10)


,course_title,subject,level,price,similarity,predicted_profit,score,url
3271,Big Data and Apache Hadoop for Developers - Fu...,Web Development,Intermediate Level,30,0.421968,3.425590e+04,0.653653,https://www.udemy.com/learn-big-data-and-apach...
2497,Web Programming with Python,Web Development,All Levels,50,0.323062,1.332244e+06,0.639717,https://www.udemy.com/web-programming-with-pyt...
3575,Display and analyze GIS data on the web with L...,Web Development,Intermediate Level,100,0.364729,2.530736e+03,0.562100,https://www.udemy.com/display-and-analyze-gis-...
3506,Fun and creative web engineering with Python a...,Web Development,All Levels,0,0.326707,-2.774360e-03,0.503260,https://www.udemy.com/web-engineering-with-pyt...
3332,Introduction to QGIS Python Programming,Web Development,Beginner Level,85,0.308342,1.696597e+04,0.476780,https://www.udemy.com/introduction-to-qgis-pyt...
3349,Implementing a Data Warehouse with SQL Server ...,Web Development,All Levels,85,0.290621,9.609041e+04,0.457920,https://www.udemy.com/implementing-a-data-ware...
2959,Projects in Django and Python,Web Development,All Levels,60,0.289195,1.051141e+05,0.456687,https://www.udemy.com/projects-in-django-and-p...
3342,Python Web Programming,Web Development,Beginner Level,100,0.270031,1.024693e+05,0.426884,https://www.udemy.com/python-web-programming/
2874,Introduction To Data Analytics Using Microsoft...,Web Development,All Levels,45,0.270105,7.893318e+04,0.424489,https://www.udemy.com/data-analytics-powerbi/
2528,Learn Python and Django: Payment Processing,Web Development,All Levels,70,0.191938,1.039788e+06,0.406546,https://www.udemy.com/learn-django-code-accept...
